# RUNBOOK: running everything on Kaggle (free 2x T4)

Total GPU time for the full set of experiments: roughly 5-6 hours. Do it over 3-4 sessions.
The same cells are in `kaggle_runbook.ipynb`; you can import that notebook into Kaggle directly.

## 0. One-time setup
1. Kaggle account -> verify your phone number (required for GPU).
2. Push this folder to a new GitHub repo (public, or private with a token).
3. Kaggle -> Create -> Notebook. In the right panel: **Accelerator = GPU T4 x2**, **Internet = On**.

## 1. Every session starts with these cells

In [2]:
# Cell A: get the code
%cd /kaggle/working
!git clone https://github.com/iamprince-IN/llm-inference-serving-hpc.git
%cd /kaggle/working/llm-inference-serving-hpc
!git pull

/kaggle/working
Cloning into 'llm-inference-serving-hpc'...
remote: Enumerating objects: 324, done.
remote: Counting objects: 100% (324/324), done.
remote: Compressing objects: 100% (253/253), done.
remote: Total 324 (delta 68), reused 320 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (324/324), 2.96 MiB | 19.54 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/kaggle/working/llm-inference-serving-hpc
Already up to date.


(No GitHub yet? Upload the zip as a Kaggle Dataset, then `!cp -r /kaggle/input/<dataset-name>/llm-inference-serving-hpc /kaggle/working/ && %cd /kaggle/working/llm-inference-serving-hpc`.)

In [4]:
# Cell B1: create a separate Python 3.12 environment and install vLLM into it (~5 min)
!python --version
!pip install -q uv
!uv venv /tmp/venv --python 3.12
%cd /kaggle/working/llm-inference-serving-hpc
!uv pip install --python /tmp/venv/bin/python vllm --torch-backend=auto
!uv pip install --python /tmp/venv/bin/python -r requirements.txt

Python 3.13.16
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: /tmp/venv
Activate with: source /tmp/venv/bin/activate
/kaggle/working/llm-inference-serving-hpc
Using Python 3.12.3 environment at: /tmp/venv
Resolved 199 packages in 1.58s                                       
Prepared 199 packages in 52.65s                                          
Installed 199 packages in 7.77s                             
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.7
 + annotated-types==0.8.0
 + anthropic==2.12.0
 + anyio==4.16.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + attrs==26.1.0
 + blake3==1.0.11
 + cachetools==8.0.2
 + cbor2==6.3.0
 + certifi==2026.11.20
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.1
 + cloudpickle==3.1.3
 + compressed-tensors==0.17.0
 + cryptography==50.0.2
 + cuda-bindings==13.5.0
 + cuda-core==1.2.2
 + cuda-pathfinder==1.9.1
 + cuda-python==13.5.0
 

In [5]:
# Cell B2: make every later "!python" and "vllm" command use the new environment
import os
os.environ["PATH"] = "/tmp/venv/bin:" + os.environ["PATH"]
os.environ["VIRTUAL_ENV"] = "/tmp/venv"
!which python && python --version
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0))"

/tmp/venv/bin/python
Python 3.12.3
vllm 0.34.0 | torch 2.14.0+cu132 | cuda True Tesla T4


In [5]:
# Cell C: record the hardware (copy the output into NOTES.md: this IS your "hardware specifications" section)
!python scripts/env_check.py

{
  "python": "3.12.3",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "torch": "2.14.0+cu132",
  "transformers": "5.22.2",
  "vllm": "0.34.0",
  "gpu": [
    "Tesla T4, 15360 MiB, 580.178.04",
    "Tesla T4, 15360 MiB, 580.178.04"
  ],
  "git_commit": "867bdea"
}
Sat Nov 21 03:17:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   0000

## 2. Day 1: feasibility (do this before anything else)

In [6]:
!python scripts/smoke_vllm.py Qwen/Qwen2.5-0.5B-Instruct
!python scripts/smoke_vllm.py Qwen/Qwen2.5-3B-Instruct
!python scripts/smoke_vllm.py Qwen/Qwen2.5-3B-Instruct-AWQ awq

W1121 03:17:50.613000 291 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1121 03:17:50.614000 291 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1121 03:17:50.614000 291 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=291) INFO 11-21 03:18:11 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}
config.json: 100%|████████████████████████████| 659/659 [00:00<00:00, 2.64MB/s]
(MainProcess pid=291) INFO 11-21 03:18:24 [model.py:761] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=291) WARNING 11-21 03:18:24 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=291) INFO 11-21 03:18:24 [model.py:2214] Using max model len 2048
(MainProcess pid=291) INFO 11-21 03:18:24 [schedule

All three must print `SMOKE TEST PASSED`. If not: copy the error to Claude. Typical fixes are an older vLLM version, or `--max-model-len 2048`.
When they pass: `!pip freeze | grep -iE "^(vllm|torch|transformers)=="` and write the versions in NOTES.md.

In [7]:
!pip freeze | grep -iE "^(vllm|torch|transformers)=="

torch==2.11.0+cu130
transformers==5.20.2


In [8]:
%cd /kaggle/working/llm-inference-serving-hpc
!git pull
!git log --oneline -3
!grep -n "wikitext" prompts/make_prompts.py
!python prompts/make_prompts.py --model Qwen/Qwen2.5-3B-Instruct

/kaggle/working/llm-inference-serving-hpc
Already up to date.
867bdea (HEAD -> main, origin/main, origin/HEAD) Merge pull request #1 from iamprince-IN/stage
b68f3f7 (origin/stage) Fix wikitext dataset id; disable prefix caching in main configs; add E5
a8d3923 Add CLAUDE.md project context
52:    text = "\n\n".join(t for t in load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")["text"] if t.strip())
README.md: 100%|██████████████████████████| 8.20k/8.20k [00:00<00:00, 16.4MB/s]

databricks-dolly-15k.jsonl: downloading bytes: █████       | 5.58MB            
databricks-dolly-15k.jsonl: downloading bytes: ████████████| 7.63MB,  754kB/s  
databricks-dolly-15k.jsonl: reconstructing file: 100%|█| 13.1MB / 13.1MB, 1.30M
Generating train split: 100%|█| 15011/15011 [00:00<00:00, 184779.62 examples/s]
dolly_200.jsonl: input tokens min=33 median=45 max=959
README.md: 100%|██████████████████████████| 10.5k/10.5k [00:00<00:00, 24.0MB/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…)

In [9]:
# quick end-to-end test of the whole benchmark pipeline (~5 min)
!python run_experiment.py configs/E3_vllm_fp16.json --users 1,8 --repeats 1 --duration 30
!python -m benchmarks.metrics --ref-users 8

[start] vllm serve Qwen/Qwen2.5-3B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.90 --port 8000 --no-enable-prefix-caching  env={'CUDA_VISIBLE_DEVICES': '0'}
[ready] http://localhost:8000 after 111s
[kv] E3_vllm_fp16_server0.log: (EngineCore pid=958) INFO 11-21 03:25:27 [utils.py:419] Using LBNHC KV cache layout.
[kv] E3_vllm_fp16_server0.log: (EngineCore pid=958) INFO 11-21 03:26:02 [gpu_worker.py:795] Available KV cache memory: 6.47 GiB
[kv] E3_vllm_fp16_server0.log: (EngineCore pid=958) INFO 11-21 03:26:33 [gpu_worker.py:1175] Free memory on device (14.41/14.56 GiB) on startup. Desired GPU memory utilization is (0.9, 13.11 GiB). Actual usage is 6.06 GiB for consumed memory (weights + non-torch), 0.57 GiB for peak activation, and 0.51 GiB for CUDAGraph memory. R
[kv] E3_vllm_fp16_server0.log: (EngineCore pid=958) INFO 11-21 03:26:35 [extensible_kv_cache.py:693] Memory after warmup: 7.75 GiB free, 0.07 GiB committed to the KV cache, 0.57 GiB kept for transient 

## 3. Main experiments (one cell each; each takes 30-60 min)

In [10]:
!python run_experiment.py configs/E0_hf_fp16.json        # baseline
!python run_experiment.py configs/E3_vllm_fp16.json      # vLLM fp16
!python run_experiment.py configs/E4_vllm_awq.json       # vLLM AWQ int4

[start] MODEL=Qwen/Qwen2.5-3B-Instruct python -m uvicorn baseline.server_hf:app --host 0.0.0.0 --port 8000  env={'CUDA_VISIBLE_DEVICES': '0'}
[ready] http://localhost:8000 after 12s
[E0_hf_fp16] users=  1 rep=0  requests=  16  errors=0  out_tok/s=    22.7  took=92s
[E0_hf_fp16] users=  1 rep=1  requests=  17  errors=0  out_tok/s=    22.8  took=96s
[E0_hf_fp16] users=  1 rep=2  requests=  16  errors=0  out_tok/s=    22.6  took=92s
[E0_hf_fp16] users=  2 rep=0  requests=  17  errors=0  out_tok/s=    22.8  took=96s
[E0_hf_fp16] users=  2 rep=1  requests=  18  errors=0  out_tok/s=    22.8  took=102s
[E0_hf_fp16] users=  2 rep=2  requests=  17  errors=0  out_tok/s=    22.6  took=97s
[E0_hf_fp16] users=  4 rep=0  requests=  20  errors=0  out_tok/s=    22.9  took=113s
[E0_hf_fp16] users=  4 rep=1  requests=  20  errors=0  out_tok/s=    23.1  took=112s
[E0_hf_fp16] users=  4 rep=2  requests=  20  errors=0  out_tok/s=    23.1  took=112s
[E0_hf_fp16] users=  8 rep=0  requests=  24  errors=0  out

## 4. Profiling the baseline (~10-15 min)

In [11]:
!python profiling/profile_baseline.py --model Qwen/Qwen2.5-3B-Instruct

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 434/434 [00:03<00:00, 126.77it/s]
params=3.09B weights=6.17GB KV/token=36.0KiB
{
  "model": "Qwen/Qwen2.5-3B-Instruct",
  "gpu": "Tesla T4",
  "params": 3085938688,
  "weight_bytes": 6171877376,
  "kv_bytes_per_token": 36864,
  "prompt_len": 512,
  "gen": 128,
  "prefill_s": 0.20263697100017453,
  "prefill_tok_per_s": 2526.6860113081684,
  "decode_tpot_median_s": 0.04221440549918043,
  "decode_tok_per_s": 23.688596065137396,
  "decode_theoretical_max_tok_per_s": 51.67031527986296,
  "MBU_decode": 0.45845658066594674,
  "MFU_decode": 0.002249278617286525,
  "MFU_prefill": 0.23991378814536257,
  "peak_bw": 320000000000.0,
  "peak_flops": 65000000000000.0,
  "time_share_prefill_for_full_request": 0.035110242369261226
}
[static batching] {'batch': 1, 'seconds': 5.558164363999822, 'out_tok_per_s': 23.029185827798614, 'peak_mem_gib': 5.8130903244018555}
[static batching] {'batch': 2

## 5. Accuracy (~30 min total)

In [12]:
!python eval/perplexity.py --model Qwen/Qwen2.5-3B-Instruct
!python eval/perplexity.py --model Qwen/Qwen2.5-3B-Instruct-AWQ --quantization awq
!python eval/output_agreement.py generate --model Qwen/Qwen2.5-3B-Instruct --tag fp16
!python eval/output_agreement.py generate --model Qwen/Qwen2.5-3B-Instruct-AWQ --quantization awq --tag awq
!python eval/output_agreement.py compare --ref fp16 --other awq

W1121 05:33:01.155000 2294 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1121 05:33:01.156000 2294 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1121 05:33:01.156000 2294 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors
(MainProcess pid=2294) INFO 11-21 05:33:11 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 1032, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-3B-Instruct'}
(MainProcess pid=2294) INFO 11-21 05:33:11 [model.py:761] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=2294) WARNING 11-21 05:33:11 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainPr

## 6. Scaling on 2 GPUs (~60 min)

In [13]:
!python run_experiment.py configs/E7a_vllm_fp16_tp2.json
!python run_experiment.py configs/E7b_vllm_fp16_2replicas.json

[start] vllm serve Qwen/Qwen2.5-3B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.90 --tensor-parallel-size 2 --port 8000 --no-enable-prefix-caching  env={'CUDA_VISIBLE_DEVICES': '0,1'}
[ready] http://localhost:8000 after 129s
[kv] E7a_vllm_fp16_tp2_server0.log: (EngineCore pid=2861) INFO 11-21 05:42:04 [utils.py:419] Using LBNHC KV cache layout.
[kv] E7a_vllm_fp16_tp2_server0.log: (Worker_TP0 pid=2883) INFO 11-21 05:42:42 [gpu_worker.py:795] Available KV cache memory: 9.7 GiB
[kv] E7a_vllm_fp16_tp2_server0.log: (Worker_TP1 pid=2884) INFO 11-21 05:43:16 [gpu_worker.py:1175] Free memory on device (14.3/14.56 GiB) on startup. Desired GPU memory utilization is (0.9, 13.11 GiB). Actual usage is 3.17 GiB for consumed memory (weights + non-torch), 0.23 GiB for peak activation, and 0.53 GiB for CUDAGraph memory. R
[kv] E7a_vllm_fp16_tp2_server0.log: (Worker_TP0 pid=2883) INFO 11-21 05:43:16 [gpu_worker.py:1175] Free memory on device (14.3/14.56 GiB) on startup. Desired 

## 7. Optional ablation (~20 min)

In [14]:
!python run_experiment.py configs/E6_vllm_fp16_eager.json

[start] vllm serve Qwen/Qwen2.5-3B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.90 --enforce-eager --port 8000 --no-enable-prefix-caching  env={'CUDA_VISIBLE_DEVICES': '0'}
[ready] http://localhost:8000 after 48s
[kv] E6_vllm_fp16_eager_server0.log: (EngineCore pid=3595) INFO 11-21 06:32:13 [utils.py:419] Using LBNHC KV cache layout.
[kv] E6_vllm_fp16_eager_server0.log: (EngineCore pid=3595) INFO 11-21 06:32:17 [gpu_worker.py:795] Available KV cache memory: 6.8 GiB
[kv] E6_vllm_fp16_eager_server0.log: (EngineCore pid=3595) INFO 11-21 06:32:20 [gpu_worker.py:1175] Free memory on device (14.41/14.56 GiB) on startup. Desired GPU memory utilization is (0.9, 13.11 GiB). Actual usage is 6.06 GiB for consumed memory (weights + non-torch), 0.24 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. R
[kv] E6_vllm_fp16_eager_server0.log: (EngineCore pid=3595) INFO 11-21 06:32:20 [extensible_kv_cache.py:693] Memory after warmup: 8.18 GiB free, 0.14 GiB committed to t

## 8. After EVERY session: make plots and download results

In [16]:
!MPLBACKEND=Agg python plots/make_plots.py
import os
os.environ["MPLBACKEND"] = "Agg"

wrote throughput_vs_users.png
wrote ttft_p95_vs_users.png
wrote tpot_vs_users.png
wrote e2e_p95_vs_users.png
wrote gpu_util_vs_users.png
wrote energy_vs_users.png
wrote ablation_cuda_graphs.png
wrote scaling_speedup_efficiency.png
wrote static_batching.png
wrote gpu_memory.png


In [17]:
!python -m benchmarks.metrics && python plots/make_plots.py
!cat results/headline_table.md
!zip -qr /kaggle/working/results_$(date +%m%d_%H%M).zip results plots

Reference load: 32 concurrent users (or the closest measured). SLO: p95 TTFT <= 1.0s

| Config | out tok/s | p95 TTFT (s) | mean TPOT (ms) | peak GPU mem (GiB) | max users within SLO | throughput x vs baseline |
|---|---|---|---|---|---|---|
| E0_hf_fp16 (u=32) | 22.9 ± 0.2 | 173.53 | 42.9 | 6.0 | 1 | 1.0x |
| E3_vllm_fp16 (u=32) | 562.2 ± 7.1 | 1.55 | 50.4 | 13.3 | 16 | 24.5x |
| E4_vllm_awq (u=32) | 857.4 ± 19.4 | 1.46 | 31.4 | 13.3 | 16 | 37.4x |
| E6_vllm_fp16_eager (u=32) | 561.0 ± 6.2 | 1.59 | 50.3 | 13.1 | 8 | 24.5x |
| E7a_vllm_fp16_tp2 (u=32) | 893.3 ± 10.8 | 0.88 | 32.6 | 12.6 | 32 | 39.0x |
| E7b_vllm_fp16_2replicas (u=32) | 766.4 ± 12.6 | 0.95 | 37.6 | 13.1 | 32 | 33.4x |

wrote results/summary_runs.csv, results/summary.csv, results/headline_table.md
wrote throughput_vs_users.png
wrote ttft_p95_vs_users.png
wrote tpot_vs_users.png
wrote e2e_p95_vs_users.png
wrote gpu_util_vs_users.png
wrote energy_vs_users.png
wrote ablation_cuda_graphs.png
wrote scaling_speedup_efficiency.

In [18]:
!python -c "import pandas as pd; s=pd.read_csv('results/summary.csv'); print(s[['config','users','out_tok_per_s_mean','ttft_p50_mean','ttft_p95_mean','tpot_mean_mean','errors_mean','littles_concurrency_mean']].round(3).to_string(index=False))"

                 config  users  out_tok_per_s_mean  ttft_p50_mean  ttft_p95_mean  tpot_mean_mean  errors_mean  littles_concurrency_mean
             E0_hf_fp16      1              22.693          0.130          0.273           0.043          0.0                     1.000
             E0_hf_fp16      2              22.734          5.736          5.963           0.043          0.0                     1.942
             E0_hf_fp16      4              23.017         16.767         17.032           0.043          0.0                     3.701
             E0_hf_fp16      8              23.113         38.761         39.048           0.043          0.0                     6.833
             E0_hf_fp16     16              23.140         82.840         83.232           0.043          0.0                    12.252
             E0_hf_fp16     32              22.928        130.239        173.531           0.043          0.0                    21.562
           E3_vllm_fp16      1              19.4

In [19]:
!python -c "import pandas as pd; r=pd.read_csv('results/summary_runs.csv'); print(r[r.users.isin([8,32])][['config','users','rep','ttft_p50','ttft_p95','ttft_p99']].round(2).to_string(index=False))"

                 config  users  rep  ttft_p50  ttft_p95  ttft_p99
             E0_hf_fp16     32    0    129.97    171.72    171.80
             E0_hf_fp16     32    1    129.65    173.81    174.33
             E0_hf_fp16     32    2    131.10    175.06    175.71
             E0_hf_fp16      8    0     38.76     39.26     39.29
             E0_hf_fp16      8    1     38.77     39.02     39.06
             E0_hf_fp16      8    2     38.75     38.87     38.96
           E3_vllm_fp16     32    0      0.83      1.67      1.77
           E3_vllm_fp16     32    1      0.86      1.44      1.67
           E3_vllm_fp16     32    2      0.93      1.53      1.54
           E3_vllm_fp16      8    0      0.32      0.63      0.89
           E3_vllm_fp16      8    1      0.29      0.42      0.54
           E3_vllm_fp16      8    2      0.37      0.60      0.60
            E4_vllm_awq     32    0      0.71      1.48      1.81
            E4_vllm_awq     32    1      0.75      1.48      1.58
          


**E5: --enable-prefix-cachinginput tokens min=33 median=45 max=959**


In [7]:
!python prompts/make_prompts.py --model Qwen/Qwen2.5-3B-Instruct
!python run_experiment.py configs/E5_vllm_fp16_prefix_cache.json

config.json: 100%|████████████████████████████| 661/661 [00:00<00:00, 2.10MB/s]
tokenizer_config.json: 100%|██████████████| 7.30k/7.30k [00:00<00:00, 14.6MB/s]
vocab.json: 100%|█████████████████████████| 2.78M/2.78M [00:00<00:00, 65.5MB/s]
merges.txt: 100%|██████████████████████████| 1.67M/1.67M [00:00<00:00, 130MB/s]
tokenizer.json: 100%|██████████████████████| 7.03M/7.03M [00:00<00:00, 155MB/s]
README.md: 100%|██████████████████████████| 8.20k/8.20k [00:00<00:00, 19.4MB/s]

databricks-dolly-15k.jsonl: downloading bytes: ███▌        | 3.92MB,  105kB/s  
databricks-dolly-15k.jsonl: downloading bytes: ████████████| 7.63MB,  732kB/s  
databricks-dolly-15k.jsonl: reconstructing file: 100%|█| 13.1MB / 13.1MB, 1.26M
Generating train split: 100%|█| 15011/15011 [00:00<00:00, 208190.97 examples/s]
dolly_200.jsonl: input tokens min=33 median=45 max=959
README.md: 100%|██████████████████████████| 10.5k/10.5k [00:00<00:00, 28.6MB/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:

In [8]:
!grep -o "enable_prefix_caching=[A-Za-z]*" results/logs/E5_vllm_fp16_prefix_cache_server0.log | head -1

enable_prefix_caching=True


In [9]:
!python -m benchmarks.metrics > /dev/null
!python -c "import pandas as pd; s=pd.read_csv('results/summary.csv'); s=s[s.config.isin(['E3_vllm_fp16','E5_vllm_fp16_prefix_cache'])&s.users.isin([1,8,32,64])]; print(s[['config','users','out_tok_per_s_mean','ttft_p50_mean','ttft_p95_mean','tpot_mean_mean']].round(3).to_string(index=False))"

                   config  users  out_tok_per_s_mean  ttft_p50_mean  ttft_p95_mean  tpot_mean_mean
             E3_vllm_fp16      1              19.438          0.060          0.192           0.051
             E3_vllm_fp16      8             224.222          0.326          0.551           0.033
             E3_vllm_fp16     32             562.197          0.872          1.546           0.050
             E3_vllm_fp16     64             820.428          1.365          2.567           0.068
E5_vllm_fp16_prefix_cache      1              34.189          0.048          0.141           0.029
E5_vllm_fp16_prefix_cache      8             243.009          0.161          0.314           0.032
E5_vllm_fp16_prefix_cache     32             673.252          0.297          0.451           0.045
E5_vllm_fp16_prefix_cache     64            1063.370          0.654          0.783           0.056


In [10]:
!zip -qr /kaggle/working/results_$(date +%m%d_%H%M).zip results plots prompts/*.jsonl

Download the zip from the Output panel (right side) and commit `results/` and `plots/` to GitHub.
**Kaggle deletes /kaggle/working when the session ends** unless you save a version.

## Long runs without babysitting
Interactive sessions can stop if the browser is idle. For long experiments, use
**Save Version -> Save & Run All (Commit)**: the notebook runs in the background, and everything in
/kaggle/working becomes the version's output. Keep only the cells you want in that run.

## If something goes wrong
- `results/logs/<config>_server0.log` holds the full server log. Read the last 30 lines: `!tail -n 30 results/logs/E3_vllm_fp16_server0.log`
- Port already in use / GPU memory still full: `!pkill -f "vllm serve"; pkill -f uvicorn; sleep 10; nvidia-smi`
- Paste errors to Claude with the log lines.